In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

rows = 5000

data = pd.DataFrame({
    'Budget': np.random.randint(20, 300, rows),
    'Runtime': np.random.randint(80, 180, rows),
    'Genre': np.random.choice(['Action', 'Comedy', 'Drama', 'SciFi', 'Horror'], rows),
    'Director_Rating': np.random.uniform(1, 10, rows),
    'Actor_Popularity': np.random.uniform(1, 10, rows),
    'Marketing_Budget': np.random.uniform(5, 100, rows),
    'Release_Month': np.random.uniform(1, 13, rows)
})

score = (
    data['Budget'] * 0.3 +
    data['Director_Rating'] * 10 +
    data['Actor_Popularity'] * 12 +
    data['Marketing_Budget'] * 1.5
)

data['Success'] = pd.cut(
    score,
    bins=[0, 120, 220, 1000],
    labels=[0, 1, 2]
)

data.to_csv('movie_success.csv', index=False)

print('Dataset created successfully!')
print(data.head())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()

from keras.models import Sequential
from keras.layers import Dense, Dropout
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.utils import to_categorical

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
data = pd.read_csv('movie_success.csv')

print(data.shape)
print(data.head())

In [ ]:
sns.countplot(x='Success', data=data)
plt.title('Movie Success Distribution')
plt.show()

In [ ]:
encoder = LabelEncoder()
data['Genre'] = encoder.fit_transform(data['Genre'])

In [ ]:
X = data.drop('Success', axis=1)
y = data['Success']

scaler = StandardScaler()
X = scaler.fit_transform(X)

y = to_categorical(y, num_classes=3)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
model = Sequential()
model.add(Dense(64, activation='relu', input_shape=(X_train.shape[1],)))
model.add(Dropout(0.3))

model.add(Dense(32, activation='relu'))
model.add(Dropout(0.3))

model.add(Dense(16, activation='relu'))
model.add(Dense(3, activation='softmax'))

In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    patience=3
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=64,
    callbacks=[early_stop, reduce_lr]
)

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.legend(['Train', 'Validation'])
plt.title('Loss')

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.legend(['Train', 'Validation'])
plt.title('Accuracy')
plt.show()

In [ ]:
predictions = model.predict(X_test)
pred_classes = np.argmax(predictions, axis=1)
actual_classes = np.argmax(y_test, axis=1)

In [ ]:
cm = confusion_matrix(actual_classes, pred_classes)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Movie Success Prediction')
plt.show()

In [ ]:
print(classification_report(actual_classes, pred_classes))

In [ ]:
new_movie = pd.DataFrame([{
    'Budget': 150,
    'Runtime': 130,
    'Genre': encoder.transform(['Action'])[0],
    'Director_Rating': 8.5,
    'Actor_Popularity': 9.0,
    'Marketing_Budget': 50,
    'Release_Month': 6
}])

new_movie_scaled = scaler.transform(new_movie)
prediction = model.predict(new_movie_scaled)
result = np.argmax(prediction)

success_labels = {0: 'Low Success', 1: 'Medium Success', 2: 'High Success'}
print(f'Predicted Success Level: {success_labels[result]}')
print(f'Confidence: {prediction[0][result]:.2%}')